# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² colorectal cancer survivors dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is provided by a Croissant schema at:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Install mlcroissant if needed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object
metadata = dataset.metadata

# Show basic metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields (columns), and their `@id`s using the Croissant schema.

In [ ]:
# List all record sets in the dataset with their @id, name, and available fields
record_sets = list(dataset.record_sets)

for rs in record_sets:
    print(f'RecordSet: {rs["@id"]}\n  Name: {getattr(rs, "name", "<unnamed>")}')
    # List columns/fields with their @id
    if hasattr(rs, "fields"):
        for field in rs.fields:
            print(f'    Field: {field["@id"]} ({getattr(field, "name", "<unnamed>")})')
    elif hasattr(rs, "columns"):
        for col in rs.columns:
            print(f'    Column: {col["@id"]} ({getattr(col, "name", "<unnamed>")})')
    print('-' * 40)

if not record_sets:
    print('No record sets found in dataset schema.')

## 3. Data Extraction
Load data from available record sets into DataFrames for analysis.

All references to record sets and fields below are by their `@id` as required by best practices.

In [ ]:
# Gather the @id for each record set from the dataset
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    print(f'Extracting records from RecordSet: {record_set_id}')
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f'Loaded DataFrame columns for {record_set_id}:', df.columns.tolist())
        print(df.head(2))
    else:
        print(f'No records found for {record_set_id}.')

if dataframes:
    chosen_record_set_id = next(iter(dataframes.keys()))  # Use the first one for further analysis
    print(f'Will use RecordSet {chosen_record_set_id} for EDA below.')
else:
    print('No usable record set with records was found.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering and normalization, referencing all columns by their `@id`.

For this demonstration, we will select a numeric field from the chosen record set for filtering and normalization, and group records by a categorical field if available.

In [ ]:
from pandas.api.types import is_numeric_dtype

# Use the primary record set loaded earlier
df = dataframes[chosen_record_set_id]

# Identify a numeric field (by @id) for demonstration
numeric_fields = [col for col in df.columns if is_numeric_dtype(df[col])]
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f'Using numeric field (@id): {numeric_field_id}')
    # Demonstrate filtering
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() != 0 else 1
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalization (z-score)
    filtered_df[numeric_field_id + "_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head())

    # Attempt to find a non-numeric (categorical) field for grouping
    group_fields = [col for col in df.columns if not is_numeric_dtype(df[col])]
    group_field_id = group_fields[0] if group_fields else None
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())
else:
    print('No numeric fields detected for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All visualizations use field `@id` references as column names.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if numeric_fields:
    plt.figure(figsize=(8,4))
    df[numeric_field_id].hist(bins=20)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean()
        group_means.plot(kind='bar')
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xlabel(group_field_id)
        plt.show()
else:
    print('No numeric fields available to plot.')

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR² CRC survivors dataset using `mlcroissant`, referencing all schema entities by their `@id`. The data is ready for further statistical or ML analysis, and the Croissant schema ensures traceable, reproducible field references for robust pipelines.